# Causal Ablation Study

This notebook tests whether the emotion direction vectors are **causally necessary** for classification, not just correlated patterns. We subtract the component of each embedding along an emotion direction and measure the accuracy drop.

**Method:** For each emotion direction $\mathbf{d}$, we compute the ablated embedding:

$$\mathbf{z}' = \mathbf{z} - (\mathbf{z} \cdot \hat{\mathbf{d}}) \hat{\mathbf{d}}$$

If removing the angry direction specifically destroys angry classification but leaves other emotions intact, the direction is causally load-bearing for that emotion. If removing **all** directions destroys overall accuracy, the directions collectively encode the model's emotion knowledge.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False

IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)

def run_command(cmd, cwd=None):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

def ensure_packages():
    pkgs = {"yaml": "pyyaml", "pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib",
            "seaborn": "seaborn", "torch": "torch", "transformers": "transformers", "sklearn": "scikit-learn", "tqdm": "tqdm"}
    missing = sorted({p for m, p in pkgs.items() if importlib.util.find_spec(m) is None})
    if missing: run_command([sys.executable, "-m", "pip", "install", "-q", *missing])
    else: print("Required packages available.")

def find_local_project_root():
    cwd = Path.cwd().resolve()
    for c in [cwd, cwd.parent, cwd / "FinalProject"]:
        c = c.resolve()
        if (c / "configs" / "wav2vec.yaml").exists() and (c / "src").exists(): return c
    raise FileNotFoundError("Could not find project root.")

def clone_clean(clean_root):
    if clean_root.exists(): shutil.rmtree(clean_root)
    run_command(["git", "clone", "--depth", "1", REPO_URL, str(clean_root)])
    return clean_root

def prepare_roots():
    rt = Path("/content") / REPO_NAME; cl = Path("/content") / f"{REPO_NAME}-clean"
    if rt.exists() and (rt / ".git").exists():
        try: run_command(["git", "-C", str(rt), "fetch", "origin"])
        except: pass
        st = subprocess.run(["git","-C",str(rt),"status","--short"],text=True,capture_output=True,check=True).stdout.splitlines()
        ab = subprocess.run(["git","-C",str(rt),"rev-list","--left-right","--count","HEAD...origin/main"],text=True,capture_output=True,check=False)
        a,b = (0,0) if ab.returncode!=0 or not ab.stdout.strip() else map(int,ab.stdout.strip().split())
        if not [l for l in st if l.strip()] and a==0:
            if b>0:
                try: run_command(["git","-C",str(rt),"pull","--ff-only","origin","main"]); cr=rt
                except: cr=clone_clean(cl)
            else: cr=rt
        else: cr=clone_clean(cl)
        return rt, cr
    elif rt.exists(): return rt, clone_clean(cl)
    else: run_command(["git","clone","--depth","1",REPO_URL,str(rt)]); return rt,rt

ensure_packages()
if IS_COLAB:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT, CODE_ROOT = prepare_roots()
else:
    PROJECT_ROOT = find_local_project_root(); CODE_ROOT = PROJECT_ROOT

os.chdir(PROJECT_ROOT)
for r in [str(CODE_ROOT), str(PROJECT_ROOT)]:
    while r in sys.path: sys.path.remove(r)
sys.path.insert(0, str(CODE_ROOT))
if str(PROJECT_ROOT) != str(CODE_ROOT): sys.path.insert(1, str(PROJECT_ROOT))
for n in [n for n in sys.modules if n == "src" or n.startswith("src.")]: del sys.modules[n]
print("Project root:", PROJECT_ROOT); print("Code root:", CODE_ROOT)

In [ ]:
experiment_name = 'wav2vec_ravdess_speaker_independent_v1'
local_checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints' / experiment_name
local_analysis_dir = PROJECT_ROOT / 'artifacts' / 'analysis' / experiment_name
local_output_dir = local_analysis_dir / 'causal_ablation'
local_output_dir.mkdir(parents=True, exist_ok=True)

drive_run_root = Path('/content/drive/MyDrive') / 'speech-emotion-directions' / 'runs' / experiment_name if IS_COLAB else None
drive_checkpoint_dir = drive_run_root / 'checkpoint' if drive_run_root else None
drive_analysis_dir = drive_run_root / 'analysis' if drive_run_root else None
drive_output_dir = drive_analysis_dir / 'causal_ablation' if drive_analysis_dir else None

checkpoint_dir = local_checkpoint_dir
if not (checkpoint_dir / 'model_state.pt').exists() and drive_checkpoint_dir and (drive_checkpoint_dir / 'model_state.pt').exists():
    checkpoint_dir = drive_checkpoint_dir
analysis_source_dir = local_analysis_dir
if not (analysis_source_dir / 'embedding_arrays.npz').exists() and drive_analysis_dir and (drive_analysis_dir / 'embedding_arrays.npz').exists():
    analysis_source_dir = drive_analysis_dir

print('Checkpoint:', checkpoint_dir)
print('Analysis:', analysis_source_dir)
print('Output:', local_output_dir)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from src.analysis.emotion_vectors import (
    build_direction_vectors, compute_class_centroids, load_embedding_artifacts,
)
from src.analysis.extract_embeddings import load_trained_checkpoint
from src.analysis.advanced_analysis import evaluate_ablation, per_class_ablation_impact

artifacts = load_embedding_artifacts(analysis_source_dir)
label_names = list(artifacts.summary['label_names'])
reference_label = 'neutral'
reference_idx = label_names.index(reference_label)
label_ids = artifacts.true_label_ids
split_array = artifacts.metadata['split'].to_numpy()
train_mask = split_array == 'train'
test_mask = split_array == 'test'

final_layer_idx = int(artifacts.layer_embeddings.shape[1] - 1)
embeddings = artifacts.layer_embeddings[:, final_layer_idx]

train_centroids = compute_class_centroids(embeddings[train_mask], label_ids[train_mask], len(label_names))
directions = build_direction_vectors(train_centroids, reference_idx)

checkpoint = load_trained_checkpoint(checkpoint_dir)
classifier_weight = checkpoint.model.classifier.weight.detach().cpu().numpy()
classifier_bias = checkpoint.model.classifier.bias.detach().cpu().numpy()

print(f'Test samples: {test_mask.sum()}, Directions shape: {directions.shape}')

## Overall Ablation Results

We test three conditions:
1. **No ablation** — baseline
2. **Single-direction ablation** — remove one emotion direction at a time
3. **All-direction ablation** — remove all 5 non-neutral directions simultaneously

In [ ]:
ablation_df = evaluate_ablation(
    embeddings=embeddings[test_mask],
    label_ids=label_ids[test_mask],
    directions=directions,
    classifier_weight=classifier_weight,
    classifier_bias=classifier_bias,
    label_names=label_names,
    reference_idx=reference_idx,
)
ablation_df.to_csv(local_output_dir / 'ablation_summary.csv', index=False)
display(ablation_df.round(4))

# Highlight key finding
baseline_f1 = ablation_df.loc[ablation_df['ablation'] == 'none', 'macro_f1'].iloc[0]
all_removed_f1 = ablation_df.loc[ablation_df['ablation'] == 'remove_all', 'macro_f1'].iloc[0]
print(f'\nRemoving all directions: Macro F1 drops from {baseline_f1:.4f} to {all_removed_f1:.4f} ({(all_removed_f1 - baseline_f1):.4f})')
print(f'This is a {((baseline_f1 - all_removed_f1) / baseline_f1 * 100):.1f}% relative drop.')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['green' if row['ablation'] == 'none' else ('red' if row['ablation'] == 'remove_all' else 'steelblue')
          for _, row in ablation_df.iterrows()]
ax.barh(ablation_df['ablation'], ablation_df['macro_f1'], color=colors)
ax.set_xlabel('Macro F1')
ax.set_title('Causal Ablation: Macro F1 After Removing Emotion Directions')
ax.axvline(baseline_f1, color='green', linestyle='--', alpha=0.5, label='Baseline')
ax.legend()
plt.tight_layout()
plt.savefig(local_output_dir / 'ablation_macro_f1.png', dpi=220)
plt.show()

## Per-Class Ablation Impact (Cross-Impact Matrix)

Does removing the *angry* direction only hurt angry classification, or does it also affect other emotions? This cross-impact matrix reveals the causal specificity of each direction.

In [ ]:
impact_df = per_class_ablation_impact(
    embeddings=embeddings[test_mask],
    label_ids=label_ids[test_mask],
    directions=directions,
    classifier_weight=classifier_weight,
    classifier_bias=classifier_bias,
    label_names=label_names,
    reference_idx=reference_idx,
)
impact_df.to_csv(local_output_dir / 'per_class_ablation_impact.csv', index=False)

# Build heatmap matrix
pivot = impact_df.pivot(index='ablated_direction', columns='evaluated_class', values='f1_delta')
display(pivot.round(4))

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=ax, linewidths=0.5)
ax.set_title('F1 Change When Removing Each Direction\n(negative = direction was causally important)')
ax.set_ylabel('Removed Direction')
ax.set_xlabel('Evaluated Emotion Class')
plt.tight_layout()
plt.savefig(local_output_dir / 'ablation_cross_impact_heatmap.png', dpi=220)
plt.show()

## Ablation Strength Sweep

How does accuracy degrade as we progressively remove more of the direction component? We scale the ablation from 0% (no removal) to 100% (full removal) to 150% (over-removal).

In [ ]:
from src.analysis.advanced_analysis import ablate_direction_component
from src.analysis.emotion_vectors import linear_classifier_probabilities, normalize_rows

test_embeddings = embeddings[test_mask]
test_labels = label_ids[test_mask]
scales = np.arange(0.0, 1.55, 0.1)

sweep_rows = []
for target_idx, target_name in enumerate(label_names):
    if target_idx == reference_idx or np.linalg.norm(directions[target_idx]) < 1e-12:
        continue
    d_hat = directions[target_idx] / np.linalg.norm(directions[target_idx])
    for scale in scales:
        ablated = test_embeddings - scale * (test_embeddings @ d_hat[:, None]) * d_hat[None, :]
        probs = linear_classifier_probabilities(ablated, classifier_weight, classifier_bias)
        preds = probs.argmax(axis=1)
        sweep_rows.append({
            'direction': target_name,
            'ablation_scale': float(scale),
            'macro_f1': float(f1_score(test_labels, preds, average='macro')),
            'target_f1': float(f1_score((test_labels == target_idx).astype(int), (preds == target_idx).astype(int))),
        })

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(local_output_dir / 'ablation_strength_sweep.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall Macro F1
for direction_name in sweep_df['direction'].unique():
    subset = sweep_df[sweep_df['direction'] == direction_name]
    axes[0].plot(subset['ablation_scale'], subset['macro_f1'], marker='o', label=direction_name)
axes[0].set_xlabel('Ablation Scale (0=none, 1=full removal)')
axes[0].set_ylabel('Macro F1')
axes[0].set_title('Overall Macro F1 vs Ablation Strength')
axes[0].legend(fontsize=8)

# Target class F1
for direction_name in sweep_df['direction'].unique():
    subset = sweep_df[sweep_df['direction'] == direction_name]
    axes[1].plot(subset['ablation_scale'], subset['target_f1'], marker='o', label=direction_name)
axes[1].set_xlabel('Ablation Scale')
axes[1].set_ylabel('Target Class F1')
axes[1].set_title('Target Emotion F1 vs Ablation Strength')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(local_output_dir / 'ablation_strength_sweep.png', dpi=220)
plt.show()

In [ ]:
# --- Drive backup ---
if not IS_COLAB:
    print('Google Drive sync is intended for Colab runtimes. Skipping.')
elif drive_output_dir is None:
    print('Drive output directory is not configured.')
else:
    import shutil
    if drive_output_dir.exists(): shutil.rmtree(drive_output_dir)
    drive_output_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_output_dir, drive_output_dir)
    print('Backed up to:', drive_output_dir)

print('\nOutput files:')
for p in sorted(local_output_dir.iterdir()):
    print(f'  {p.name}')